# 🕵️ Dia 39 — Coletando e enriquecendo IOCs sem tocar no adversário

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 39* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-39.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Você ia rodar `dig coopvale-pix.example` por reflexo. Aí lembrou do log ilustrativo do Dia 32: o servidor do kit registrou duas visitas do IP da Vigia no dia 5, e no dia 6 esse IP passou a receber página em branco. Rafa só apontou para o `.bash_history`: *"A partir de hoje, este arquivo é auditado."*

Neste notebook você vai:

1. **Escrever `restaurar()`, `neutralizar()` e um extrator de IOCs** com `re` e `ipaddress`.
2. **Registrar a procedência** do anexo (hora UTC + SHA-256).
3. **Montar a tabela de enriquecimento** a partir de respostas simuladas dos serviços do dia.
4. **Fazer uma consulta passiva real (RDAP)**, com plano B, e conferir que o alvo é o serviço.
5. **Jogar "ativo ou passivo?"** e rodar o **auditor de histórico**.

🐍 Só biblioteca padrão do Python. Nada para instalar. Funciona offline: toda célula que tenta a rede tem plano B embutido. Rode as células de cima para baixo (Shift+Enter).

## 1. O anexo de IOCs e a procedência (passo 1 do original)

O anexo do ALR-2026-01 está embutido abaixo (igual a `caso-continuo/dados/dia-39/anexo-iocs-ALR-2026-01.txt`). Antes de qualquer coisa: **de onde veio, quando você pegou, qual o hash**.

In [ ]:
import hashlib                          # SHA-256 do anexo: prova de qual versão você usou
from datetime import datetime, timezone  # hora UTC da coleta

ANEXO = '''# Anexo de IOCs do alerta ALR-2026-01 — 100% FICTÍCIO (Operação Isca, Investigador Cyber 90 Dias, Dia 39)
# Domínios .example (RFC 2606), IPs de documentação (RFC 5737). TLP:AMBER.
# Datas de atividade: 2026-10-05 a 2026-10-11 (BRT). Indicadores já defangados, como manda o Dia 39.

Domínios imitação observados nos SMS e nas capturas:
  hxxps://serraazul-pix[.]example/
  serraazul-seguranca[.]example
  coopvale-pix[.]example
  coopvale-pix[.]example  (repetido no relatório da CoopVale)
  unicoop-sul-pix[.]example

Encurtador usado no SMS da Serra Azul:
  hxxps://lnk[.]example/sa7

Hospedagem observada:
  203.0.113[.]77   (serraazul-pix e coopvale-pix)
  198.51.100[.]40  (credminas-seguranca e unicoop-sul-pix)

Artefatos:
  SHA-256 do isca.js (Serra Azul): c6a500d420b2dd0197d73e8fa5ebe43f81ebfa3e86e56b91a9e909a4beae08b6
  SHA-256 do arquivo vazio que aparece na lista de recursos da página: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
'''

coletado_em = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%MZ")
sha_anexo = hashlib.sha256(ANEXO.encode("utf-8")).hexdigest()

PROCEDENCIA = f"""Seed report: ALR-2026-01, Vigia + cooperativas (fictício), 2026-10-12, TLP:AMBER
IOC file:    caso-continuo/dados/dia-39/anexo-iocs-ALR-2026-01.txt (SHA-256 {sha_anexo[:16]}...)
Retrieved:   {coletado_em}
Atividade:   2026-10-05 a 2026-10-11 (BRT), segundo o próprio anexo"""
print(PROCEDENCIA)

## 2. Restaurar, extrair, deduplicar, neutralizar (passo 2 do original, sem CyberChef)

- `restaurar()` = *Fang URL* do CyberChef (`hxxp` → `http`, `[.]` → `.`);
- `neutralizar()` = *Defang* (só o último ponto vira `[.]`, como no original: `203.0.113[.]7`);
- o extrator usa expressões regulares e o módulo `ipaddress` para **validar** cada IP (um `999.1.1.1` não passa).

In [ ]:
import ipaddress   # valida IPs e diz se são privados, de documentação etc.
import re          # expressões regulares para achar padrões no texto

def restaurar(texto):
    """Fang: volta ao valor real, só para processar. Nunca cole o resultado num relatório."""
    return texto.replace("[.]", ".").replace("hxxp", "http")

def neutralizar(valor):
    """Defang: troca o ÚLTIMO ponto por [.] e http por hxxp. É o que vai para o documento."""
    valor = valor.replace("http", "hxxp")
    return "[.]".join(valor.rsplit(".", 1))

RE_SHA256 = re.compile(r"\b[a-f0-9]{64}\b")
RE_IPV4 = re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}\b")
RE_URL = re.compile(r"https?://[^\s)]+")
RE_DOMINIO = re.compile(r"\b(?:[a-z0-9-]+\.)+example\b")   # neste laboratório, só .example

def extrair(texto):
    """Devolve um dicionário de conjuntos (já deduplicados) por tipo de IOC."""
    t = restaurar(texto)
    ips = set()
    for candidato in RE_IPV4.findall(t):
        try:
            ips.add(str(ipaddress.ip_address(candidato)))   # só entra se for um IP válido
        except ValueError:
            pass
    return {"url": set(RE_URL.findall(t)), "dominio": set(RE_DOMINIO.findall(t)),
            "ipv4": ips, "sha256": set(RE_SHA256.findall(t))}

IOCS = extrair(ANEXO)
for tipo, valores in IOCS.items():
    print(f"{tipo:<8} {len(valores)} únicos:", sorted(neutralizar(v) for v in valores))
print("\n(5 linhas de domínio no anexo → 4 domínios únicos + o encurtador: a duplicata da CoopVale sumiu.)")

## 3. As respostas dos serviços (passos 3 e 4 do original)

Sem chave de API e sem encostar no adversário: o arquivo `respostas-servicos-simuladas.json` traz o que Team Cymru, Shodan InternetDB, GreyNoise, AbuseIPDB, ThreatFox, RDAP, VirusTotal e urlscan.io **teriam** respondido, nos formatos reais, com valores fictícios. Embutido abaixo.

In [ ]:
import json

RESPOSTAS = json.loads('''{
 "_aviso": "Respostas SIMULADAS e FICTÍCIAS de serviços de consulta passiva (Operação Isca, Dia 39). Formatos inspirados nas APIs reais citadas no dia (Team Cymru, Shodan InternetDB, GreyNoise Community, AbuseIPDB, ThreatFox, RDAP, VirusTotal, urlscan.io), mas nenhum valor veio dessas APIs. ASNs de documentação (RFC 5398). Hora da 'consulta': 2026-10-12.",
 "consultado_em": "2026-10-12T13:10:00Z",
 "ip": {
  "203.0.113.77": {
   "team_cymru": "AS      | IP           | BGP Prefix       | CC | Registry | Allocated  | AS Name\\n64501   | 203.0.113.77 | 203.0.113.64/26  | BR | lacnic   | 2021-08-10 | HOSTBARATO-EXEMPLO, BR",
   "internetdb": {
    "cpes": [],
    "hostnames": [
     "serraazul-pix.example",
     "coopvale-pix.example"
    ],
    "ip": "203.0.113.77",
    "ports": [
     22,
     80,
     443
    ],
    "tags": [
     "shared-hosting"
    ],
    "vulns": []
   },
   "greynoise": {
    "ip": "203.0.113.77",
    "noise": false,
    "riot": false,
    "classification": "unknown",
    "message": "IP not observed scanning the internet or contained in RIOT data set.",
    "last_seen": null
   },
   "abuseipdb": {
    "abuseConfidenceScore": 34,
    "countryCode": "BR",
    "isp": "Hostbarato Exemplo",
    "usageType": "Data Center/Web Hosting/Transit",
    "totalReports": 6,
    "lastReportedAt": "2026-10-09T21:40:12Z"
   },
   "threatfox": [
    {
     "ioc": "203.0.113.77",
     "threat_type": "payload_delivery",
     "malware_printable": "Unknown malware",
     "first_seen": "2026-10-06 11:02:00 UTC",
     "last_seen": null,
     "confidence_level": 50
    }
   ],
   "passive_dns_contagem": 41,
   "nota": "41 domínios resolvem para este IP em passive DNS (hospedagem compartilhada)."
  },
  "198.51.100.40": {
   "team_cymru": "AS      | IP            | BGP Prefix       | CC | Registry | Allocated  | AS Name\\n64502   | 198.51.100.40 | 198.51.100.0/26  | BR | lacnic   | 2023-02-14 | DNSBARATO-EXEMPLO, BR",
   "internetdb": {
    "cpes": [],
    "hostnames": [
     "credminas-seguranca.example",
     "unicoop-sul-pix.example"
    ],
    "ip": "198.51.100.40",
    "ports": [
     443
    ],
    "tags": [],
    "vulns": []
   },
   "greynoise": {
    "ip": "198.51.100.40",
    "noise": false,
    "riot": false,
    "classification": "unknown",
    "message": "IP not observed scanning the internet or contained in RIOT data set.",
    "last_seen": null
   },
   "abuseipdb": {
    "abuseConfidenceScore": 0,
    "countryCode": "BR",
    "isp": "DNSbarato Exemplo",
    "usageType": "Data Center/Web Hosting/Transit",
    "totalReports": 0,
    "lastReportedAt": null
   },
   "threatfox": [],
   "passive_dns_contagem": 2,
   "nota": "Só 2 domínios em passive DNS: os dois do kit."
  },
  "203.0.113.78": {
   "team_cymru": "AS      | IP           | BGP Prefix       | CC | Registry | Allocated  | AS Name\\n64501   | 203.0.113.78 | 203.0.113.64/26  | BR | lacnic   | 2021-08-10 | HOSTBARATO-EXEMPLO, BR",
   "internetdb": {
    "cpes": [],
    "hostnames": [
     "serraazul-seguranca.example",
     "mail.serraazul-seguranca.example"
    ],
    "ip": "203.0.113.78",
    "ports": [
     25,
     443
    ],
    "tags": [],
    "vulns": []
   },
   "greynoise": {
    "ip": "203.0.113.78",
    "noise": false,
    "riot": false,
    "classification": "unknown",
    "message": "IP not observed scanning the internet or contained in RIOT data set.",
    "last_seen": null
   },
   "abuseipdb": {
    "abuseConfidenceScore": 0,
    "countryCode": "BR",
    "isp": "Hostbarato Exemplo",
    "usageType": "Data Center/Web Hosting/Transit",
    "totalReports": 0,
    "lastReportedAt": null
   },
   "threatfox": [],
   "passive_dns_contagem": 2,
   "nota": "Surgiu na resolução passiva de serraazul-seguranca.example. NÃO estava no anexo do alerta."
  }
 },
 "dominio": {
  "serraazul-pix.example": {
   "rdap": {
    "handle": "FICT-0001",
    "events": [
     {
      "eventAction": "registration",
      "eventDate": "2026-10-02T14:22:10Z"
     },
     {
      "eventAction": "expiration",
      "eventDate": "2027-10-02T14:22:10Z"
     }
    ],
    "nameservers": [
     "ns1.dnsbarato.example",
     "ns2.dnsbarato.example"
    ]
   },
   "vt": {
    "creation_date": "2026-10-02",
    "registrar": "Registrar Barato (fictício)",
    "last_analysis_stats": {
     "malicious": 9,
     "suspicious": 2,
     "harmless": 58,
     "undetected": 22
    },
    "categories": {
     "Fornecedor A": "phishing"
    }
   },
   "vt_resolutions": [
    [
     "2026-10-04T19:30:11Z",
     "203.0.113.77"
    ]
   ],
   "urlscan": [
    [
     "2026-10-05T10:42:18Z",
     "203.0.113.77",
     "HOSTBARATO-EXEMPLO",
     "Serra Azul - Regularização Pix"
    ]
   ]
  },
  "serraazul-seguranca.example": {
   "rdap": {
    "handle": "FICT-0002",
    "events": [
     {
      "eventAction": "registration",
      "eventDate": "2026-10-02T14:25:47Z"
     }
    ],
    "nameservers": [
     "ns1.dnsbarato.example"
    ]
   },
   "vt": {
    "creation_date": "2026-10-02",
    "registrar": "Registrar Barato (fictício)",
    "last_analysis_stats": {
     "malicious": 4,
     "suspicious": 1,
     "harmless": 61,
     "undetected": 25
    },
    "categories": {}
   },
   "vt_resolutions": [
    [
     "2026-10-05T08:01:44Z",
     "203.0.113.78"
    ]
   ],
   "urlscan": []
  },
  "coopvale-pix.example": {
   "rdap": {
    "handle": "FICT-0005",
    "events": [
     {
      "eventAction": "registration",
      "eventDate": "2026-10-07T03:11:09Z"
     }
    ],
    "nameservers": [
     "ns1.dnsbarato.example",
     "ns2.dnsbarato.example"
    ]
   },
   "vt": {
    "creation_date": "2026-10-07",
    "registrar": "Registrar Barato (fictício)",
    "last_analysis_stats": {
     "malicious": 6,
     "suspicious": 1,
     "harmless": 60,
     "undetected": 24
    },
    "categories": {
     "Fornecedor A": "phishing"
    }
   },
   "vt_resolutions": [
    [
     "2026-10-07T22:15:00Z",
     "203.0.113.77"
    ]
   ],
   "urlscan": [
    [
     "2026-10-08T09:03:51Z",
     "203.0.113.77",
     "HOSTBARATO-EXEMPLO",
     "CoopVale - Regularização Pix"
    ]
   ]
  },
  "unicoop-sul-pix.example": {
   "rdap": {
    "handle": "FICT-0008",
    "events": [
     {
      "eventAction": "registration",
      "eventDate": "2026-10-09T17:40:02Z"
     }
    ],
    "nameservers": [
     "ns1.dnsbarato.example"
    ]
   },
   "vt": {
    "creation_date": "2026-10-09",
    "registrar": "Registrar Barato (fictício)",
    "last_analysis_stats": {
     "malicious": 2,
     "suspicious": 0,
     "harmless": 63,
     "undetected": 26
    },
    "categories": {}
   },
   "vt_resolutions": [
    [
     "2026-10-10T06:20:33Z",
     "198.51.100.40"
    ]
   ],
   "urlscan": [
    [
     "2026-10-10T12:44:09Z",
     "198.51.100.40",
     "DNSBARATO-EXEMPLO",
     "Unicoop Sul - Regularização Pix"
    ]
   ]
  },
  "lnk.example": {
   "rdap": {
    "handle": "FICT-0099",
    "events": [
     {
      "eventAction": "registration",
      "eventDate": "2015-06-01T00:00:00Z"
     }
    ],
    "nameservers": [
     "ns1.encurtador.example"
    ]
   },
   "vt": {
    "creation_date": "2015-06-01",
    "registrar": "Registrar Tradicional (fictício)",
    "last_analysis_stats": {
     "malicious": 0,
     "suspicious": 0,
     "harmless": 80,
     "undetected": 11
    },
    "categories": {
     "Fornecedor A": "url shortener"
    }
   },
   "vt_resolutions": [
    [
     "2026-10-01T00:00:00Z",
     "192.0.2.200"
    ]
   ],
   "urlscan": [],
   "nota": "Encurtador público usado por milhares de sites legítimos."
  }
 }
}
''')
print(RESPOSTAS["_aviso"])
print("\nIPs com resposta:", list(RESPOSTAS["ip"]))
print("Domínios com resposta:", list(RESPOSTAS["dominio"]))

## 4. A tabela de enriquecimento (passo 5)

Uma linha por **indicador × fonte × campo**, com a hora em que a fonte observou (quando ela informa) e a hora da sua coleta. A coluna final, "ainda útil para detecção?", é decisão sua: a célula sugere uma regra simples, e você pode sobrescrever.

In [ ]:
COLETA = RESPOSTAS["consultado_em"]
TABELA = []   # cada linha: dicionário com as colunas do original

def linha(ioc, tipo, como, fonte, campo, valor, obs="n/d"):
    valor_doc = ioc if tipo == "sha256" else neutralizar(ioc)   # hash não tem ponto para neutralizar
    TABELA.append({"ioc": valor_doc, "tipo": tipo, "no_relatorio": como, "fonte": fonte,
                   "campo": campo, "valor": str(valor), "obs_fonte": obs, "obtido": COLETA})

PAPEL = {"203.0.113.77": "hospedagem do kit", "198.51.100.40": "hospedagem do kit",
         "203.0.113.78": "(não estava no anexo)"}
for ip, r in RESPOSTAS["ip"].items():
    como = PAPEL.get(ip, "?")
    asn = r["team_cymru"].splitlines()[1].split("|")
    linha(ip, "ipv4", como, "Team Cymru", "ASN / org", f"AS{asn[0].strip()} {asn[-1].strip()}")
    linha(ip, "ipv4", como, "Shodan InternetDB", "portas / hostnames", f"{r['internetdb']['ports']} {r['internetdb']['hostnames']}")
    linha(ip, "ipv4", como, "GreyNoise", "noise / classification", f"{r['greynoise']['noise']} / {r['greynoise']['classification']}")
    ab = r["abuseipdb"]
    linha(ip, "ipv4", como, "AbuseIPDB", "score / denúncias", f"{ab['abuseConfidenceScore']} / {ab['totalReports']}", ab["lastReportedAt"] or "n/d")
    tf = r["threatfox"]
    linha(ip, "ipv4", como, "ThreatFox", "registro", tf[0]["threat_type"] if tf else "sem registro", tf[0]["first_seen"] if tf else "n/d")
    linha(ip, "ipv4", como, "passive DNS", "nº de domínios no IP", r["passive_dns_contagem"])

for dom, r in RESPOSTAS["dominio"].items():
    como = "encurtador do SMS" if dom == "lnk.example" else "domínio imitação"
    reg = next((e["eventDate"] for e in r["rdap"]["events"] if e["eventAction"] == "registration"), "n/d")
    linha(dom, "domain", como, "RDAP", "registro / NS", f"{reg[:10]} / {', '.join(r['rdap']['nameservers'])}")
    linha(dom, "domain", como, "VirusTotal", "detecções maliciosas", r["vt"]["last_analysis_stats"]["malicious"])
    for quando, ip in r["vt_resolutions"]:
        linha(dom, "domain", como, "VT resolutions (passive DNS)", "resolveu para", neutralizar(ip), quando)
    for quando, ip, asn, titulo in r["urlscan"]:
        linha(dom, "domain", como, "urlscan.io (busca)", "título da página", titulo, quando)

for h in IOCS["sha256"]:
    vazio = h == hashlib.sha256(b"").hexdigest()      # o hash do arquivo vazio, calculado na hora
    linha(h[:16] + "...", "sha256", "recurso da página", "cálculo local", "é o arquivo vazio?", "SIM" if vazio else "não")

print(f"{len({l['ioc'] for l in TABELA})} indicadores, {len(TABELA)} linhas.")

In [ ]:
# "Ainda útil para detecção?" — regra sugerida. Sobrescreva em DECISAO_MANUAL se discordar.
def util(ioc_neutro):
    real = restaurar(ioc_neutro)
    if real in RESPOSTAS["ip"] and RESPOSTAS["ip"][real]["passive_dns_contagem"] > 10:
        return "não", "hospedagem compartilhada: muitos domínios benignos no mesmo IP"
    if real == "lnk.example":
        return "não", "encurtador público usado por milhares de sites legítimos (bloquear só a URL /sa7)"
    if real.startswith("e3b0c44298fc1c14"):
        return "não", "SHA-256 do arquivo vazio: casa com milhares de sites sem relação"
    if real in RESPOSTAS["dominio"]:
        return "sim", "domínio dedicado ao golpe, registrado dias antes do disparo"
    return "desconhecido", "pouca informação; reavaliar no Dia 40"

DECISAO_MANUAL = {}   # ex.: {"198.51.100[.]40": ("sim", "só 2 domínios, os dois do kit")}

decisoes = {}
for l in TABELA:
    decisoes.setdefault(l["ioc"], DECISAO_MANUAL.get(l["ioc"]) or util(l["ioc"]))

cab = "| IOC | Tipo | No relatório como | Fonte | Campo | Valor | Obs. da fonte | Obtido (UTC) |"
md_tab = [cab, "|" + "---|" * 8]
for l in TABELA:
    md_tab.append("| " + " | ".join([l["ioc"], l["tipo"], l["no_relatorio"], l["fonte"], l["campo"],
                                     l["valor"], l["obs_fonte"], l["obtido"]]) + " |")
md_util = ["| IOC | Ainda útil para detecção? | Por quê |", "|---|---|---|"]
md_util += [f"| {i} | {d[0]} | {d[1]} |" for i, d in decisoes.items()]
print("\n".join(md_util))

### O indicador novo

O original mostra como o *passive DNS* revela um segundo IP. Aqui: qual IP aparece nas resoluções e **não** estava no anexo?

In [ ]:
ips_anexo = IOCS["ipv4"]
novos = set()
for dom, r in RESPOSTAS["dominio"].items():
    for quando, ip in r["vt_resolutions"]:
        if ip not in ips_anexo and dom != "lnk.example":   # o encurtador é público, não conta
            novos.add((ip, dom, quando))
for ip, dom, quando in sorted(novos):
    print(f"🆕 {neutralizar(ip)} — revelado pelas resoluções do VT de {neutralizar(dom)} em {quando}")
print("→ Semente de pivô para o Dia 40.")

with open("enriquecimento-alr-2026-01.md", "w", encoding="utf-8") as f:
    f.write("# Enriquecimento — ALR-2026-01\n\n```\n" + PROCEDENCIA + "\n```\n\n" + "\n".join(md_tab)
            + "\n\n## Ainda útil para detecção?\n\n" + "\n".join(md_util)
            + "\n\n## Sementes do Dia 40\n\n" + "\n".join(f"- {neutralizar(i)} ({neutralizar(d)}, {q})" for i, d, q in sorted(novos)) + "\n")
print("💾 enriquecimento-alr-2026-01.md salvo.")

## 5. Uma consulta passiva de verdade (RDAP)

RDAP não pede chave. A célula consulta um domínio **legítimo e público** (por padrão, `registro.br`, no próprio RDAP do Registro.br) só para você ver o formato real. Domínios `.example` não existem no RDAP de verdade, por isso o caso usa as respostas simuladas.

Repare na checagem `alvo_e_servico()`: antes de qualquer requisição, ela confere que o **host da URL é o serviço de consulta**, e não o indicador. É o checkpoint do original transformado em código.

In [ ]:
#@title Consulta RDAP { display-mode: "form" }
DOMINIO_PUBLICO = "registro.br"  #@param {type:"string"}

import urllib.parse
import urllib.request

SERVICOS = {"rdap.registro.br", "rdap.org", "internetdb.shodan.io", "api.greynoise.io",
            "api.abuseipdb.com", "threatfox-api.abuse.ch", "www.virustotal.com", "urlscan.io"}
INDICADORES = {restaurar(l["ioc"]) for l in TABELA}

def alvo_e_servico(url):
    """True só se o host da URL for um serviço de consulta conhecido e não um indicador do caso."""
    host = urllib.parse.urlparse(url).hostname or ""
    return host in SERVICOS and host not in INDICADORES

url = f"https://rdap.registro.br/domain/{DOMINIO_PUBLICO.strip()}"
assert alvo_e_servico(url), "Essa URL contataria algo que não é um serviço de consulta!"
try:
    with urllib.request.urlopen(url, timeout=20) as r:
        dado = json.load(r)
    origem = "rede (rdap.registro.br)"
except Exception as erro:
    # Plano B: um esqueleto no formato RDAP (RFC 9083), com valores fictícios
    dado = {"ldhName": DOMINIO_PUBLICO, "events": [{"eventAction": "registration", "eventDate": "(exemplo offline)"}],
            "nameservers": [{"ldhName": "a.dns.br (exemplo offline)"}]}
    origem = f"plano B offline ({erro.__class__.__name__})"
print("Origem:", origem)
print({"dominio": dado.get("ldhName"),
       "eventos": [(e.get("eventAction"), e.get("eventDate")) for e in dado.get("events", [])],
       "nameservers": [n.get("ldhName") for n in dado.get("nameservers", [])]})

# E a URL proibida? Teste:
print("\nhttps://coopvale-pix.example/ passa na checagem?", alvo_e_servico("https://coopvale-pix.example/"))

## 6. 🎮 Quiz: ativo ou passivo?

**S** = passivo (pode), **N** = ativo (não pode, contata o adversário).

In [ ]:
MODO_JOGO = True
QUIZ = [
    ("dig coopvale-pix.example", False, "Seu resolvedor pergunta ao servidor de nomes DO GOLPISTA."),
    ("curl https://internetdb.shodan.io/203.0.113.77", True, "O host é o Shodan. O IP é só o parâmetro."),
    ("Abrir hxxps://serraazul-pix[.]example num navegador 'só para ver'", False, "O servidor do kit registra seu IP (Dia 32)."),
    ("Buscar page.domain:coopvale-pix.example no urlscan.io", True, "Buscar varreduras existentes não varre nada."),
    ("Submeter a URL do kit para uma NOVA varredura pública no urlscan", False, "Manda o scanner ao adversário e publica seu interesse."),
    ("Resoluções do VirusTotal (passive DNS) do domínio", True, "Histórico que o VT já coletou."),
    ("ping 198.51.100.40", False, "Pacote seu chega ao servidor do adversário."),
    ("RDAP do domínio no rdap.org", True, "Pergunta ao registro, não ao domínio."),
]
acertos = 0
for i, (acao, passivo, porque) in enumerate(QUIZ, 1):
    print(f"\n[{i}/{len(QUIZ)}] {acao}")
    if MODO_JOGO:
        r = ""
        while r not in ("S", "N"):
            r = input("   Passivo, pode? (S/N): ").strip().upper()
        ok = (r == "S") == passivo
        acertos += ok
        print("   ✅" if ok else "   ❌", end=" ")
    print("PASSIVO" if passivo else "ATIVO", "—", porque)
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 7. 🔎 O auditor de histórico (checkpoint do original)

O checkpoint pede: procure no histórico por `dig`, `ping`, `nslookup` em indicadores, e por `curl` cujo **host** é o indicador. Cole o seu `.bash_history` em `HISTORICO` (ou use o exemplo, que tem dois deslizes).

In [ ]:
HISTORICO = '''mkdir -p ~/cti-lab/day39 && cd ~/cti-lab/day39
sort -u domains.txt -o domains.txt
curl -s https://internetdb.shodan.io/203.0.113.77 | jq .
curl -sL https://rdap.org/domain/coopvale-pix.example | jq .
dig coopvale-pix.example
curl -s "https://urlscan.io/api/v1/search/?q=page.domain:unicoop-sul-pix.example&size=10"
curl -sI https://serraazul-pix.example/
whois -h whois.cymru.com " -v 198.51.100.40"'''

FERRAMENTAS_ATIVAS = ("dig", "nslookup", "ping", "host", "traceroute", "nc", "wget")

def auditar(historico, indicadores):
    alertas = []
    for n, cmd in enumerate(historico.splitlines(), 1):
        partes = cmd.split()
        if not partes:
            continue
        toca = [i for i in indicadores if i in cmd]          # o comando menciona algum indicador?
        if partes[0] in FERRAMENTAS_ATIVAS and toca:
            alertas.append((n, cmd, f"{partes[0]} direto no indicador"))
        for u in re.findall(r"https?://[^\s\"']+", cmd):     # cada URL do comando
            host = urllib.parse.urlparse(u).hostname or ""
            if host in indicadores:
                alertas.append((n, cmd, f"curl/navegador com host = indicador ({neutralizar(host)})"))
    return alertas

achados = auditar(HISTORICO, INDICADORES)
if achados:
    for n, cmd, motivo in achados:
        print(f"⛔ linha {n}: {motivo}\n     {neutralizar(cmd)}")
else:
    print("✅ Nenhum comando ativo contra indicadores.")

In [ ]:
try:
    from google.colab import files
    files.download("enriquecimento-alr-2026-01.md")
except ImportError:
    print("(Fora do Colab: enriquecimento-alr-2026-01.md ficou na pasta atual.)")

## ✅ Antes de fechar o Dia 39

- [ ] Procedência com hora UTC e SHA-256 do anexo
- [ ] Tabela com 10+ indicadores e 30+ linhas, cada uma com fonte e duas horas
- [ ] Pelo menos um "não útil para detecção" com motivo (o hash vazio, o encurtador, o IP compartilhado)
- [ ] O indicador novo anotado como semente do Dia 40
- [ ] Auditor de histórico limpo no **seu** histórico

**Amanhã (Dia 40):** pivotar a partir dessas sementes até montar o grafo de infraestrutura do CA-ISCA-01, com cada aresta dizendo **qual pivô** a produziu.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>